# Stage 4 — Readability & Faithfulness Analysis

**Reads:** `outputs/explanations.json`  
**Writes:** `outputs/results.csv`

Computes per-explanation metrics:

| Metric | Description |
|--------|-------------|
| `flesch_reading_ease` | Higher = simpler text |
| `flesch_kincaid_grade` | Higher = more complex |
| `smog_index` | Estimated years of education needed |
| `lime_coverage` | Fraction of LIME features referenced in explanation |
| `ontology_hit_rate` | Fraction of features mapped to ≥1 ancestor |

> **Tip:** This notebook is fast — safe to re-run freely as you tweak display cells.

## 1. Imports

In [1]:
import pandas as pd

from config import ANALYSIS_RESULTS_PATH, EXPLANATIONS_PATH
from pipeline_helpers import (
    checkpoint_exists,
    lime_coverage,
    load_checkpoint,
    ontology_hit_rate,
    readability_metrics,
    save_checkpoint,
)

## 2. Configuration

In [2]:
# Set True to recompute even if results.csv already exists
FORCE_RERUN = False

## 3. Load Stage 3 output

In [3]:
if not EXPLANATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Explanations not found at '{EXPLANATIONS_PATH}'.\n"
        "Please run 03_llm.ipynb first."
    )
data = load_checkpoint(EXPLANATIONS_PATH)
print(f"Loaded {len(data)} explanations.")

[Checkpoint] Loaded 1 records ← 'outputs\explanations.json'
Loaded 1 explanations.


## 4. Compute metrics

In [4]:
rows = []
for item in data:
    explanation  = item.get("explanation", "")
    feature_data = item.get("feature_data", [])

    row = {
        "text":              item["text"][:80] + "…",
        "predicted_class":   item["predicted_class"],
        "confidence":        item["confidence"],
        "user_category":     item["user_category"],
        "ablation_mode":     item["ablation_mode"],
        "explanation":       explanation,
        "lime_coverage":     lime_coverage(explanation, feature_data),
        "ontology_hit_rate": ontology_hit_rate(feature_data),
        **readability_metrics(explanation),
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(f"✅ Metrics computed for {len(df)} explanations.")

✅ Metrics computed for 1 explanations.


## 5. Full results table

In [5]:
pd.set_option("display.max_colwidth", 60)
df[[
    "predicted_class", "user_category", "ablation_mode",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "lime_coverage", "ontology_hit_rate"
]]

,predicted_class,user_category,ablation_mode,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
0,Cardiovascular diseases,EXPERT,normal,2.14,17.5,18.2,1.0,1.0


## 6. Summary — mean metrics by user category & ablation mode

In [6]:
metric_cols = [
    "flesch_reading_ease", "flesch_kincaid_grade",
    "smog_index", "lime_coverage", "ontology_hit_rate",
]
summary = (
    df.groupby(["user_category", "ablation_mode"])[metric_cols]
    .mean()
    .round(3)
)
summary

,,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
user_category,ablation_mode,,,,,
EXPERT,normal,2.14,17.5,18.2,1.0,1.0


## 7. [Optional] Per-class breakdown

In [7]:
df.groupby("predicted_class")[metric_cols].mean().round(3)

,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
predicted_class,,,,,
Cardiovascular diseases,2.14,17.5,18.2,1.0,1.0


## 8. [Optional] Read a specific explanation in full

In [8]:
# Change the index to read any explanation in full
IDX = 0

row = df.iloc[IDX]
print(f"Text     : {row['text']}")
print(f"Class    : {row['predicted_class']} ({row['confidence']})")
print(f"User     : {row['user_category']}")
print(f"Ablation : {row['ablation_mode']}")
print(f"\n── Explanation ────────────────────────────────────")
print(row["explanation"])

Text     : Intravenous nicardipine: an effective new agent for the treatment of severe hype…
Class    : Cardiovascular diseases (0.9417)
User     : EXPERT
Ablation : normal

── Explanation ────────────────────────────────────
assistant
The user's input is related to cardiovascular diseases, specifically mentioning hypertension as a relevant factor. The prediction indicates that the user has a high likelihood of having cardiovascular diseases. The LIME analysis highlights hypertension as a significant feature, which is part of the broader category of cardiovascular diseases. This means that individuals who have hypertension are more likely to develop other types of cardiovascular diseases such as arterial diseases and vascular diseases.


## 9. Save to CSV

In [9]:
df.to_csv(ANALYSIS_RESULTS_PATH, index=False)
print(f"✅ Saved {len(df)} rows → '{ANALYSIS_RESULTS_PATH}'")

✅ Saved 1 rows → 'outputs\results.csv'
